# Bayesian SSE Association Models

In [1]:
from __future__ import annotations

from collections.abc import Iterable, Sequence
from pathlib import Path
import re
import sys

import os
os.environ["JAX_PLATFORMS"] = "cpu"

import arviz as az
import bambi as bmb
from IPython.display import display
import numpy as np
import pandas as pd
from scipy.special import logit as logit_func

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib import (  # noqa: E402
    DEFAULT_MIXING_FEATURES,
    add_observed_mixing_entropy_scales,
    HIGH_PRIORITY_CANDIDATE_TIERS,
    load_sse_outputs,
    load_sequence_data,
)

SSE_OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [2]:
STANDARDISE_SPECS = {
    "z_wn_prop_sequenced": "wn_prop_sequenced",
    "z_log1p_wn_positive_tests": "log1p_wn_positive_tests",
    "z_dz_cum_prop_sequenced": "dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita": "dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity": "dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests": "log1p_dz_cum_positive_tests",
    "z_dz_cum_prop_vaccinated": "dz_cum_prop_vaccinated",
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "20-24",
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": 1,
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "z_dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests",
]


def add_standardised_adjusters(data: pd.DataFrame) -> pd.DataFrame:
    """Add standardised surveillance and context adjusters used by notebooks."""
    out = add_observed_mixing_entropy_scales(data)
    if "wn_positive_tests" in out.columns:
        out["log1p_wn_positive_tests"] = np.log1p(out["wn_positive_tests"])
    if "dz_cum_positive_tests" in out.columns:
        out["log1p_dz_cum_positive_tests"] = np.log1p(out["dz_cum_positive_tests"])
    for target, source in STANDARDISE_SPECS.items():
        if source not in out.columns:
            continue
        values = out[source].astype(float)
        sd = values.std(skipna=True)
        if pd.isna(sd) or sd == 0:
            out[target] = np.nan
        else:
            out[target] = (values - values.mean(skipna=True)) / sd
    return out

In [3]:
RANDOM_SEED = 123
CLUSTER_ID_COL = "cluster_id"
MIXING_GROUP_VARS = ("window_idx", "clade")
COMP_GROUP_VARS = MIXING_GROUP_VARS + (CLUSTER_ID_COL,)

RESULT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "bayesian_socio_geo_demo"

COMPOSITION_PREDICTORS = {
    spec["column"]: spec["reference"] for spec in COMPOSITION_SPECS
}
MIXING_PREDICTORS = list(DEFAULT_MIXING_FEATURES)
EXPANDED_ADJUSTERS = list(EXPANDED_CONTEXT_ADJUSTERS)

# Narrow these lists for a smoke-test grid, e.g. ["sex"] or ["sex_entropy_z"].
COMPOSITION_GRID_PREDICTORS = list(COMPOSITION_PREDICTORS)
MIXING_GRID_PREDICTORS = list(MIXING_PREDICTORS)

# Test-run controls. When USE_DEV_SAMPLE=True, model fitting uses balanced-ish
# samples created by sample_model_test_data; the full complete-case frames are
# still retained for reporting and for later full-data runs.
USE_DEV_SAMPLE = False
DEV_SAMPLE_ROWS = 1_000
COMPOSITION_DEV_SAMPLE_ROWS = DEV_SAMPLE_ROWS * 5
MIXING_DEV_SAMPLE_ROWS = DEV_SAMPLE_ROWS
COMPOSITION_POSITIVE_FRACTION = 0.25
MIXING_POSITIVE_FRACTION = 0.047

RUN_COMPOSITION_GRID = True
RUN_MIXING_GRID = True

SAVE_INFERENCE_DATA = False

SAMPLING_KWARGS = {
    "draws": 2_000,
    "tune": 2_000,
    "chains": 4,
    "cores": 4,
    "target_accept": 0.99,
    "random_seed": RANDOM_SEED,
    "inference_method": "nutpie",
}

## Load and Align Data

The node outcome follows the main pipeline: high-priority burst or burden candidates among nodes at least as large as the smallest high-priority candidate. Sequence-level models inherit the candidate label from their cluster.


In [4]:
sse_outputs = load_sse_outputs(SSE_OUTPUT_DIR)
cluster_data = add_standardised_adjusters(sse_outputs.cluster_table.copy())
sequence_data = add_standardised_adjusters(load_sequence_data())

cluster_data["candidate"] = cluster_data["candidate_tier"].isin(
    HIGH_PRIORITY_CANDIDATE_TIERS
)

candidate_sizes = cluster_data.loc[cluster_data["candidate"], "cluster_size"].dropna()
if candidate_sizes.empty:
    raise ValueError("No high-priority candidate nodes were found.")

min_candidate_size = int(candidate_sizes.min())
eligible_nodes = cluster_data.loc[
    cluster_data["cluster_size"].ge(min_candidate_size)
].copy()

eligible_sequence_data = sequence_data.merge(
    eligible_nodes[[CLUSTER_ID_COL, "candidate"]],
    on=CLUSTER_ID_COL,
    how="inner",
)

candidate_node_rate = float(eligible_nodes["candidate"].mean())
candidate_sequence_rate = float(eligible_sequence_data["candidate"].mean())

candidate_summary = pd.DataFrame(
    [
        {
            "dataset": "eligible_nodes",
            "rows": len(eligible_nodes),
            "candidate_rate": candidate_node_rate,
            "candidates": int(eligible_nodes["candidate"].sum()),
        },
        {
            "dataset": "eligible_sequence_data",
            "rows": len(eligible_sequence_data),
            "candidate_rate": candidate_sequence_rate,
            "candidates": int(eligible_sequence_data["candidate"].sum()),
        },
    ]
)
display(candidate_summary)

,dataset,rows,candidate_rate,candidates
0,eligible_nodes,13059,0.047017,614
1,eligible_sequence_data,264139,0.253049,66840


## Data and Formula Helpers


In [6]:
def _unique_preserve_order(items: Iterable[str]) -> list[str]:
    """Return unique strings in first-seen order."""
    out: list[str] = []
    seen: set[str] = set()
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out


def _categories_from(series: pd.Series) -> list:
    """Use declared categories when present; otherwise preserve observed order."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        return list(series.cat.categories)
    return series.dropna().drop_duplicates().tolist()


def sample_model_test_data(
    data: pd.DataFrame,
    *,
    outcome: str = "candidate",
    max_rows: int = DEV_SAMPLE_ROWS,
    positive_fraction: float = 0.35,
    random_state: int = RANDOM_SEED,
    categorical_vars: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Return a balanced-ish development sample covering observed categories."""
    positives = data.loc[data[outcome] == 1]
    negatives = data.loc[data[outcome] == 0]

    if positives.empty or negatives.empty:
        raise ValueError(f"'{outcome}' must contain at least one 0 and one 1.")

    positive_fraction = float(np.clip(positive_fraction, 0.01, 0.99))
    n_pos = min(len(positives), max(1, int(round(max_rows * positive_fraction))))
    n_neg = min(len(negatives), max_rows - n_pos)
    if n_neg <= 0:
        raise ValueError("Sampling settings left no room for negative controls.")

    category_cols = [
        col
        for col in _unique_preserve_order(categorical_vars or ())
        if col in data.columns
    ]
    category_levels = {
        col: [
            level
            for level in _categories_from(data[col])
            if data[col].eq(level).any()
        ]
        for col in category_cols
    }

    missing_by_col = {col: set(levels) for col, levels in category_levels.items()}
    seed_indices: list[object] = []
    selected_indices: set[object] = set()

    for col, levels in category_levels.items():
        for level in levels:
            if level not in missing_by_col[col]:
                continue

            candidates = data.loc[data[col].eq(level)]
            coverage = pd.Series(0, index=candidates.index, dtype=int)
            for cover_col, missing_levels in missing_by_col.items():
                if missing_levels:
                    coverage += candidates[cover_col].isin(missing_levels).astype(int)

            best_indices = coverage.loc[coverage.eq(coverage.max())].index
            seed_row = data.loc[best_indices].sample(
                n=1,
                random_state=random_state + len(seed_indices),
                replace=False,
            )
            idx = seed_row.index[0]
            if idx not in selected_indices:
                seed_indices.append(idx)
                selected_indices.add(idx)

            row = data.loc[idx]
            for cover_col in category_cols:
                missing_by_col[cover_col].discard(row[cover_col])

    if len(seed_indices) > max_rows:
        raise ValueError(
            "max_rows is too small to include all observed categorical levels "
            f"({len(seed_indices)} seed rows needed; max_rows={max_rows})."
        )

    sampled_parts = [data.loc[seed_indices]] if seed_indices else []
    seed_outcomes = (
        data.loc[seed_indices, outcome] # type: ignore
        if seed_indices
        else pd.Series(dtype=int)
    )
    seed_pos = int((seed_outcomes == 1).sum()) # type: ignore
    seed_neg = int((seed_outcomes == 0).sum()) # type: ignore
    remaining_capacity = max_rows - len(seed_indices)

    n_pos_fill = max(0, n_pos - seed_pos)
    n_neg_fill = max(0, n_neg - seed_neg)
    if n_pos_fill + n_neg_fill > remaining_capacity:
        if seed_pos >= n_pos:
            n_pos_fill = 0
            n_neg_fill = remaining_capacity
        elif seed_neg >= n_neg:
            n_neg_fill = 0
            n_pos_fill = remaining_capacity
        else:
            scale = remaining_capacity / (n_pos_fill + n_neg_fill)
            n_pos_fill = int(round(n_pos_fill * scale))
            n_neg_fill = remaining_capacity - n_pos_fill

    def sample_available(frame: pd.DataFrame, n: int, seed_offset: int) -> pd.DataFrame:
        available = frame.drop(index=list(selected_indices), errors="ignore")
        n = min(n, len(available))
        if n <= 0:
            return available.iloc[0:0]
        out = available.sample(
            n=n,
            random_state=random_state + seed_offset,
            replace=False,
        )
        selected_indices.update(out.index.tolist())
        return out

    sampled_parts.append(sample_available(positives, n_pos_fill, 10_000))
    sampled_parts.append(sample_available(negatives, n_neg_fill, 20_000))

    sampled = pd.concat(sampled_parts, axis=0)
    if len(sampled) < max_rows:
        top_up = sample_available(data, max_rows - len(sampled), 30_000)
        sampled = pd.concat([sampled, top_up], axis=0)

    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    for col in category_cols:
        sampled[col] = pd.Categorical(
            sampled[col],
            categories=_categories_from(data[col]),
        )

    return sampled


def get_complete_case_data(
    df: pd.DataFrame,
    *,
    outcome: str = "candidate",
    predictors: Iterable[str] | None = None,
    group_vars: Sequence[str] = MIXING_GROUP_VARS,
    categorical_vars: Sequence[str] = (),
    id_cols: Sequence[str] = (CLUSTER_ID_COL,),
    verbose: bool = True,
) -> pd.DataFrame:
    """Create a complete-case model frame with stable categorical dtypes."""
    if predictors is None:
        raise ValueError("Please provide predictor columns.")

    required_cols = _unique_preserve_order(
        [outcome, *id_cols, *list(predictors), *group_vars]
    )
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    before_n = len(df)
    out = df.loc[:, required_cols].dropna().copy()
    after_n = len(out)

    out[outcome] = out[outcome].astype(int)

    categorical_cols = _unique_preserve_order(
        [*categorical_vars, *group_vars, *id_cols]
    )
    for col in categorical_cols:
        if col in out.columns:
            out[col] = pd.Categorical(out[col], categories=_categories_from(df[col]))

    if verbose:
        print("Complete-case summary")
        print("---------------------")
        print(f"Rows before dropna: {before_n:,}")
        print(f"Rows after dropna:  {after_n:,}")
        print(f"Rows dropped:       {before_n - after_n:,}")
        print(f"Percent retained:   {after_n / before_n:.1%}")
        print(f"Candidate rate:     {out[outcome].mean():.2%}")
        print("\nGrouping levels:")
        for col in group_vars:
            print(f"{col}: {out[col].nunique()} levels")

    return out


def treatment_term(variable: str, reference: str | int | float | None) -> str:
    """Return a Bambi/formulae categorical term with an optional reference level."""
    if reference is None:
        return f"C({variable})"
    return f"C({variable}, Treatment(reference={reference!r}))"


def formula_with_varying_intercepts(
    outcome: str,
    terms: str | Sequence[str],
    *,
    group_vars: Sequence[str] = MIXING_GROUP_VARS,
) -> str:
    """Build a logistic mixed-model formula with window/clade intercepts."""
    if isinstance(terms, str):
        terms = [terms]
    rhs_terms = [*terms, *(f"(1|{group})" for group in group_vars)]
    return f"{outcome} ~ " + " + ".join(rhs_terms)


def compact_categorical_levels(
    data: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    """Drop unobserved categories for sampled grouping variables."""
    out = data.copy()
    for col in columns:
        if col not in out.columns:
            continue
        if isinstance(out[col].dtype, pd.CategoricalDtype):
            out[col] = out[col].cat.remove_unused_categories()
        else:
            out[col] = pd.Categorical(out[col])
    return out


def make_model_fit_data(
    data: pd.DataFrame,
    positive_fraction: float,
    *,
    label: str,
    use_dev_sample: bool,
    max_rows: int,
    categorical_vars: Sequence[str] = (),
    compact_category_vars: Sequence[str] = (),
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Return either the full model frame or a sampled frame for smoke tests."""
    if not use_dev_sample:
        print(f"{label}: using full frame ({len(data):,} rows).")
        return data.copy()

    sampled = sample_model_test_data(
        data,
        max_rows=max_rows,
        positive_fraction=positive_fraction,
        random_state=random_state,
        categorical_vars=categorical_vars,
    )
    sampled = compact_categorical_levels(sampled, compact_category_vars)
    print(
        f"{label}: using development sample "
        f"({len(sampled):,} of {len(data):,} rows; "
        f"candidate rate {sampled['candidate'].mean():.2%})."
    )
    return sampled


def model_frame_summary_row(
    label: str,
    full_df: pd.DataFrame,
    fit_df: pd.DataFrame,
    *,
    outcome: str = "candidate",
) -> dict[str, object]:
    """Summarise the full and fitting frames side by side."""
    return {
        "frame": label,
        "full_rows": len(full_df),
        "fit_rows": len(fit_df),
        "fit_fraction": len(fit_df) / len(full_df) if len(full_df) else np.nan,
        "full_candidates": int(full_df[outcome].sum()),
        "fit_candidates": int(fit_df[outcome].sum()),
        "full_candidate_rate": float(full_df[outcome].mean()),
        "fit_candidate_rate": float(fit_df[outcome].mean()),
        "use_dev_sample": len(fit_df) != len(full_df),
    }


## Bambi Fitting and Summaries


In [7]:
_RANDOM_EFFECT_RE = re.compile(r"\([^|()]+\|([^()]+)\)")


def random_effect_groups(formula: str) -> list[str]:
    """Extract grouping variables from terms such as ``(1|window_idx)``."""
    return [match.strip() for match in _RANDOM_EFFECT_RE.findall(formula)]


def fit_bayesian_logistic_model(
    data: pd.DataFrame,
    formula: str,
    *,
    family: str = "bernoulli",
    categorical: Sequence[str] | None = None,
    fixed_prior_sigma: float = 1.0,
    intercept_prior_sigma: float = 1.5,
    random_effect_sigma: float = 1.0,
    draws: int = 2_000,
    tune: int = 2_000,
    chains: int = 4,
    cores: int = 4,
    target_accept: float = 0.99,
    inference_method: str = "pymc",
    random_seed: int = RANDOM_SEED,
    log_likelihood: bool = True,
    noncentered: bool = True,
) -> tuple[bmb.Model, az.InferenceData]:
    """Fit a Bambi hierarchical logistic regression with weakly informative priors."""
    if "~" not in formula:
        raise ValueError("Formula must contain '~', e.g. 'candidate ~ x + (1|group)'.")

    response = formula.split("~", 1)[0].strip()
    if response not in data.columns:
        raise ValueError(f"Response variable '{response}' not found in data.")

    outcome_mean = float(np.clip(data[response].mean(), 1e-6, 1 - 1e-6))
    priors = {
        "Intercept": bmb.Prior(
            "Normal",
            mu=logit_func(outcome_mean),
            sigma=intercept_prior_sigma,
        ),
        "common": bmb.Prior("Normal", mu=0, sigma=fixed_prior_sigma),
        "group_specific": bmb.Prior(
            "Normal",
            mu=0,
            sigma=bmb.Prior("HalfNormal", sigma=random_effect_sigma),
        ),
    }

    group_cols = random_effect_groups(formula)
    categorical_cols = _unique_preserve_order([*(categorical or ()), *group_cols])

    model = bmb.Model(
        formula=formula,
        data=data,
        family=family,
        priors=priors,
        categorical=categorical_cols or None,
        noncentered=noncentered,
    )
    fit_kwargs = {
        "draws": draws,
        "tune": tune,
        "chains": chains,
        "cores": cores,
        "inference_method": inference_method,
        "target_accept": target_accept,
        "random_seed": random_seed,
    }
    idata = model.fit(**fit_kwargs)
    if log_likelihood:
        try:
            model.compute_log_likelihood(idata)
        except Exception as e:
            print(f"Error computing log likelihood: {e}")
    return model, idata


def _print_section(title: str, char: str = "=") -> None:
    print(f"\n{title}")
    print(char * len(title))


def _format_df_for_print(
    df: pd.DataFrame,
    *,
    float_digits: int = 4,
    width: int = 160,
    max_colwidth: int = 80,
) -> str:
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        width,
        "display.max_colwidth",
        max_colwidth,
        "display.float_format",
        lambda x: f"{x:,.{float_digits}f}",
    ):
        return df.to_string()


def _show_table(
    df: pd.DataFrame,
    *,
    display_tables: bool = False,
    float_digits: int = 4,
) -> None:
    if display_tables:
        display(df.style.format(precision=float_digits))
    else:
        print(_format_df_for_print(df, float_digits=float_digits))


def _available_posterior_vars(
    idata: az.InferenceData,
    var_names: Sequence[str] | None,
) -> list[str] | None:
    if var_names is None:
        return None
    available = set(idata.posterior.data_vars)  # type: ignore
    selected = [var for var in var_names if var in available]
    missing = [var for var in var_names if var not in available]
    if missing:
        print("Skipping unavailable posterior variables:")
        for var in missing:
            print(f"  - {var}")
    if not selected:
        raise KeyError("None of the requested posterior variables were found.")
    return selected


def summarise_bambi_idata(
    idata: az.InferenceData,
    *,
    var_names: Sequence[str] | None = None,
    ci_prob: float = 0.95,
    odds_ratio_vars: Sequence[str] | None = None,
    print_diagnostics: bool = True,
    rhat_threshold: float = 1.01,
    ess_threshold: int = 400,
    display_tables: bool = False,
    float_digits: int = 4,
) -> pd.DataFrame:
    """Print diagnostics and return a focused ArviZ posterior summary."""
    selected_vars = _available_posterior_vars(idata, var_names)
    summary = az.summary(
        idata,
        var_names=selected_vars,
        ci_prob=ci_prob,
        ci_kind="hdi",
    )

    diagnostic_summary = az.summary(
        idata,
        ci_prob=ci_prob,
        ci_kind="hdi",
    )
    if print_diagnostics:
        _print_section("Bayesian model diagnostics")
        diagnostic_rows = []

        if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:  # type: ignore
            n_div = int(idata.sample_stats["diverging"].sum().item())  # type: ignore
            n_total = int(idata.sample_stats["diverging"].size)  # type: ignore
            div_rate = n_div / n_total
            diagnostic_rows.append(
                {
                    "Diagnostic": "Divergences",
                    "Value": f"{n_div} / {n_total} ({div_rate:.2%})",
                    "Status": "OK" if n_div == 0 else "WARNING",
                    "Interpretation": (
                        "No divergent transitions."
                        if n_div == 0
                        else "Investigate divergent transitions."
                    ),
                }
            )
        try:
            energy = idata.sample_stats["energy"]
            bfmi_result = az.bfmi(energy)

            if hasattr(bfmi_result, "values"):
                bfmi = np.asarray(bfmi_result.values).astype(float).ravel()
            else:
                bfmi = np.asarray(bfmi_result).astype(float).ravel()

            bfmi = bfmi[np.isfinite(bfmi)]

            if bfmi.size == 0:
                raise ValueError("BFMI returned no finite values.")

            min_bfmi = float(np.nanmin(bfmi))
            bfmi_by_chain = ", ".join(f"{x:.3f}" for x in bfmi)

            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": f"min={min_bfmi:.3f}; chains=[{bfmi_by_chain}]",
                    "Status": "OK" if min_bfmi >= 0.3 else "WARNING",
                    "Interpretation": (
                        "Energy exploration looks acceptable."
                        if min_bfmi >= 0.3
                        else "One or more chains have BFMI < 0.3."
                    ),
                }
            )

        except Exception as exc:
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": "Could not compute",
                    "Status": "NA",
                    "Interpretation": str(exc),
                }
            )    
        if "r_hat" in diagnostic_summary.columns:
            max_rhat = float(diagnostic_summary["r_hat"].max(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max R-hat",
                    "Value": f"{max_rhat:.4f}",
                    "Status": "OK" if max_rhat <= rhat_threshold else "WARNING",
                    "Interpretation": (
                        f"All posterior variables are at or below {rhat_threshold}."
                        if max_rhat <= rhat_threshold
                        else f"Some posterior variables exceed {rhat_threshold}."
                    ),
                }
            )

        if "ess_bulk" in diagnostic_summary.columns:
            min_bulk_ess = float(diagnostic_summary["ess_bulk"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min bulk ESS",
                    "Value": f"{min_bulk_ess:.1f}",
                    "Status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All bulk ESS values are at least {ess_threshold}."
                        if min_bulk_ess >= ess_threshold
                        else f"Some bulk ESS values are below {ess_threshold}."
                    ),
                }
            )

        if "ess_tail" in diagnostic_summary.columns:
            min_tail_ess = float(diagnostic_summary["ess_tail"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min tail ESS",
                    "Value": f"{min_tail_ess:.1f}",
                    "Status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All tail ESS values are at least {ess_threshold}."
                        if min_tail_ess >= ess_threshold
                        else f"Some tail ESS values are below {ess_threshold}."
                    ),
                }
            )

        if hasattr(idata, "sample_stats") and "tree_depth" in idata.sample_stats:  # type: ignore
            max_tree_depth = int(idata.sample_stats["tree_depth"].max().item())  # type: ignore
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max tree depth",
                    "Value": str(max_tree_depth),
                    "Status": "INFO",
                    "Interpretation": "Maximum observed tree depth.",
                }
            )

        _show_table(
            pd.DataFrame(diagnostic_rows),
            display_tables=display_tables,
            float_digits=float_digits,
        )

        _print_section("Posterior summary")
        _show_table(
            summary,  # type: ignore
            display_tables=display_tables,
            float_digits=float_digits,
        )

    if odds_ratio_vars is not None:
        _print_section("Odds-ratio summaries")
        available = set(idata.posterior.data_vars)  # type: ignore
        for var in odds_ratio_vars:
            if var not in available:
                print(f"\n{var}: not found in idata.posterior")
                continue

            beta = idata.posterior[var]  # type: ignore
            odds_ratio = np.exp(beta)
            or_idata = odds_ratio.to_dataset(name=f"OR_{var}")
            or_summary = az.summary(
                or_idata,
                ci_prob=ci_prob,
                ci_kind="hdi",
            )

            prob_df = pd.DataFrame(
                {
                    "Quantity": [
                        "P(beta > 0 | data)",
                        "P(beta < 0 | data)",
                        "P(OR > 1 | data)",
                        "P(OR < 1 | data)",
                    ],
                    "Probability": [
                        float((beta > 0).mean().item()),
                        float((beta < 0).mean().item()),
                        float((odds_ratio > 1).mean().item()),
                        float((odds_ratio < 1).mean().item()),
                    ],
                }
            )

            _print_section(var, char="-")
            _show_table(
                or_summary,  # type: ignore
                display_tables=display_tables,
                float_digits=float_digits,
            )
            _show_table(
                prob_df,
                display_tables=display_tables,
                float_digits=float_digits,
            )

    return summary  # type: ignore


def fit_and_summarise_model(
    data: pd.DataFrame,
    formula: str,
    *,
    var_names: Sequence[str],
    odds_ratio_vars: Sequence[str],
    categorical: Sequence[str] = MIXING_GROUP_VARS,
    display_tables: bool = True,
    **fit_kwargs,
) -> dict[str, object]:
    """Fit a model and keep the model, posterior, formula, and focused summary."""
    model, idata = fit_bayesian_logistic_model(
        data=data,
        formula=formula,
        categorical=categorical,
        **fit_kwargs,
    )
    summary = summarise_bambi_idata(
        idata,
        var_names=var_names,
        ci_prob=0.95,
        odds_ratio_vars=odds_ratio_vars,
        display_tables=display_tables,
    )
    response = formula.split("~", 1)[0].strip()
    return {
        "formula": formula,
        "model": model,
        "idata": idata,
        "summary": summary,
        "var_names": list(var_names),
        "odds_ratio_vars": list(odds_ratio_vars),
        "n_rows": len(data),
        "candidate_rate": float(data[response].mean()),
    }


## Mixing Models: Node-Level Association

Mixing models ask whether candidate nodes have unusual cluster-level entropy or context profiles, again with varying intercepts for window and clade.

In [8]:
unknown_mixing_predictors = sorted(set(MIXING_GRID_PREDICTORS) - set(MIXING_PREDICTORS))
if unknown_mixing_predictors:
    raise ValueError(f"Unknown MIXING_GRID_PREDICTORS: {unknown_mixing_predictors}")

mixing_primary_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=MIXING_GRID_PREDICTORS,
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_expanded_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=[*MIXING_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_primary_fit_df = make_model_fit_data(
    mixing_primary_df,
    label="mixing_primary",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=MIXING_DEV_SAMPLE_ROWS,
    positive_fraction=MIXING_POSITIVE_FRACTION,
    compact_category_vars=MIXING_GROUP_VARS,
)
mixing_expanded_fit_df = make_model_fit_data(
    mixing_expanded_df,
    label="mixing_expanded",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=MIXING_DEV_SAMPLE_ROWS,
    positive_fraction=MIXING_POSITIVE_FRACTION,
    compact_category_vars=MIXING_GROUP_VARS,
)

mixing_fit_frame_summary = pd.DataFrame(
    [
        model_frame_summary_row(
            "mixing_primary", mixing_primary_df, mixing_primary_fit_df
        ),
        model_frame_summary_row(
            "mixing_expanded", mixing_expanded_df, mixing_expanded_fit_df
        ),
    ]
)
display(mixing_fit_frame_summary)

mixing_single_primary_formulas = {
    col: formula_with_varying_intercepts("candidate", col)
    for col in MIXING_GRID_PREDICTORS
}
mixing_single_expanded_formulas = {
    col: formula_with_varying_intercepts(
        "candidate",
        [col, *EXPANDED_ADJUSTERS],
    )
    for col in MIXING_GRID_PREDICTORS
}
joint_mixing_primary_formula = formula_with_varying_intercepts(
    "candidate",
    MIXING_GRID_PREDICTORS,
)
joint_mixing_expanded_formula = formula_with_varying_intercepts(
    "candidate",
    [*MIXING_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
)

mixing_model_grid = pd.DataFrame(
    [
        {"model": "single_primary", "predictor": col, "formula": formula}
        for col, formula in mixing_single_primary_formulas.items()
    ]
    + [
        {"model": "single_expanded", "predictor": col, "formula": formula}
        for col, formula in mixing_single_expanded_formulas.items()
    ]
    + [
        {
            "model": "joint_primary",
            "predictor": "all_mixing_predictors",
            "formula": joint_mixing_primary_formula,
        },
        {
            "model": "joint_expanded",
            "predictor": "all_mixing_predictors",
            "formula": joint_mixing_expanded_formula,
        },
    ]
)
display(mixing_model_grid)


Complete-case summary
---------------------
Rows before dropna: 13,059
Rows after dropna:  12,525
Rows dropped:       534
Percent retained:   95.9%
Candidate rate:     4.82%

Grouping levels:
window_idx: 58 levels
clade: 19 levels
Complete-case summary
---------------------
Rows before dropna: 13,059
Rows after dropna:  12,525
Rows dropped:       534
Percent retained:   95.9%
Candidate rate:     4.82%

Grouping levels:
window_idx: 58 levels
clade: 19 levels
mixing_primary: using full frame (12,525 rows).
mixing_expanded: using full frame (12,525 rows).


,frame,full_rows,fit_rows,fit_fraction,full_candidates,fit_candidates,full_candidate_rate,fit_candidate_rate,use_dev_sample
0,mixing_primary,12525,12525,1.0,604,604,0.048224,0.048224,False
1,mixing_expanded,12525,12525,1.0,604,604,0.048224,0.048224,False


,model,predictor,formula
0,single_primary,sex_entropy_z,candidate ~ sex_entropy_z + (1|window_idx) + (...
1,single_primary,age_entropy_z,candidate ~ age_entropy_z + (1|window_idx) + (...
2,single_primary,simd_entropy_z,candidate ~ simd_entropy_z + (1|window_idx) + ...
3,single_primary,datazone_entropy_z,candidate ~ datazone_entropy_z + (1|window_idx...
4,single_primary,local_authority_entropy_z,candidate ~ local_authority_entropy_z + (1|win...
5,single_primary,urban_rural_entropy_z,candidate ~ urban_rural_entropy_z + (1|window_...
6,single_primary,health_board_entropy_z,candidate ~ health_board_entropy_z + (1|window...
7,single_primary,vaccination_entropy_z,candidate ~ vaccination_entropy_z + (1|window_...
8,single_expanded,sex_entropy_z,candidate ~ sex_entropy_z + z_dz_cum_prop_sequ...
9,single_expanded,age_entropy_z,candidate ~ age_entropy_z + z_dz_cum_prop_sequ...


In [9]:
single_primary_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_primary_formulas.items():
        print(f"\nFitting primary mixing model: {col}")
        single_primary_results[col] = fit_and_summarise_model(
            data=mixing_primary_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
        break  # Remove or adjust this break to run the full grid.
else:
    print("RUN_MIXING_GRID=False; skipping primary single-predictor sampling.")


Modeling the probability that candidate==1



Fitting primary mixing model: sex_entropy_z


NUTS[nutpie]: [Intercept, sex_entropy_z, 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset]


Output()


Bayesian model diagnostics


...


Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,-3.12,0.14,-3.5,-2.9,791,928,1.00,0.0053,0.0054
sex_entropy_z,-0.0753,0.0181,-0.11,-0.039,10242,6171,1.00,0.00018,0.00013
1|window_idx_sigma,0.0402,0.0311,0.0013,0.12,3751,2620,1.00,0.00041,0.00036
1|clade_sigma,0.21,0.16,0.014,0.61,692,1021,1.00,0.0058,0.0069



Odds-ratio summaries

sex_entropy_z
-------------


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
OR_sex_entropy_z,0.9277,0.0168,0.9,0.96,10242,6171,1.00,0.00017,0.00012


,Quantity,Probability
0,P(beta > 0 | data),0.0000
1,P(beta < 0 | data),1.0000
2,P(OR > 1 | data),0.0000
3,P(OR < 1 | data),1.0000


In [ ]:
single_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_expanded_formulas.items():
        print(f"\nFitting expanded mixing model: {col}")
        single_expanded_results[col] = fit_and_summarise_model(
            data=mixing_expanded_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                *EXPANDED_ADJUSTERS,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_MIXING_GRID=False; skipping expanded single-predictor sampling.")


In [ ]:
joint_primary_results: dict[str, dict[str, object]] = {}
joint_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    print("\nFitting joint primary mixing model")
    joint_primary_results["joint_primary"] = fit_and_summarise_model(
        data=mixing_primary_fit_df,
        formula=joint_mixing_primary_formula,
        var_names=[
            "Intercept",
            *MIXING_GRID_PREDICTORS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_GRID_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )

    print("\nFitting joint expanded mixing model")
    joint_expanded_results["joint_expanded"] = fit_and_summarise_model(
        data=mixing_expanded_fit_df,
        formula=joint_mixing_expanded_formula,
        var_names=[
            "Intercept",
            *MIXING_GRID_PREDICTORS,
            *EXPANDED_ADJUSTERS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_GRID_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    print("RUN_MIXING_GRID=False; skipping joint mixing-model sampling.")


## Composition Models: Sequence-Level Association

Composition models ask whether individual sequence attributes are associated with membership in a candidate cluster, with varying intercepts for window and clade.


In [ ]:
unknown_composition_predictors = sorted(
    set(COMPOSITION_GRID_PREDICTORS) - set(COMPOSITION_PREDICTORS)
)
if unknown_composition_predictors:
    raise ValueError(
        f"Unknown COMPOSITION_GRID_PREDICTORS: {unknown_composition_predictors}"
    )

composition_terms = {
    column: treatment_term(column, COMPOSITION_PREDICTORS[column])
    for column in COMPOSITION_GRID_PREDICTORS
}
composition_categorical_vars = (*COMP_GROUP_VARS, *COMPOSITION_GRID_PREDICTORS)

composition_primary_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=COMPOSITION_GRID_PREDICTORS,
    group_vars=COMP_GROUP_VARS,
    categorical_vars=composition_categorical_vars,
)

composition_expanded_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=[*COMPOSITION_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
    group_vars=COMP_GROUP_VARS,
    categorical_vars=composition_categorical_vars,
)

composition_primary_fit_df = make_model_fit_data(
    composition_primary_df,
    label="composition_primary",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=COMPOSITION_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    categorical_vars=COMPOSITION_GRID_PREDICTORS,
    compact_category_vars=COMP_GROUP_VARS,
)
composition_expanded_fit_df = make_model_fit_data(
    composition_expanded_df,
    label="composition_expanded",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=COMPOSITION_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    categorical_vars=COMPOSITION_GRID_PREDICTORS,
    compact_category_vars=COMP_GROUP_VARS,
)

composition_fit_frame_summary = pd.DataFrame(
    [
        model_frame_summary_row(
            "composition_primary", composition_primary_df, composition_primary_fit_df
        ),
        model_frame_summary_row(
            "composition_expanded", composition_expanded_df, composition_expanded_fit_df
        ),
    ]
)
display(composition_fit_frame_summary)

composition_single_primary_formulas = {
    column: formula_with_varying_intercepts(
        "candidate", term, group_vars=COMP_GROUP_VARS
    )
    for column, term in composition_terms.items()
}
composition_single_expanded_formulas = {
    column: formula_with_varying_intercepts(
        "candidate",
        [term, *EXPANDED_ADJUSTERS],
        group_vars=COMP_GROUP_VARS,
    )
    for column, term in composition_terms.items()
}
joint_composition_primary_formula = formula_with_varying_intercepts(
    "candidate",
    list(composition_terms.values()),
    group_vars=COMP_GROUP_VARS,
)
joint_composition_expanded_formula = formula_with_varying_intercepts(
    "candidate",
    [*composition_terms.values(), *EXPANDED_ADJUSTERS],
    group_vars=COMP_GROUP_VARS,
)

composition_model_df = composition_primary_fit_df
composition_formulas = composition_single_primary_formulas
joint_composition_formula = joint_composition_primary_formula

composition_model_grid = pd.DataFrame(
    [
        {"model": "single_primary", "predictor": column, "formula": formula}
        for column, formula in composition_single_primary_formulas.items()
    ]
    + [
        {"model": "single_expanded", "predictor": column, "formula": formula}
        for column, formula in composition_single_expanded_formulas.items()
    ]
    + [
        {
            "model": "joint_primary",
            "predictor": "all_composition_predictors",
            "formula": joint_composition_primary_formula,
        },
        {
            "model": "joint_expanded",
            "predictor": "all_composition_predictors",
            "formula": joint_composition_expanded_formula,
        },
    ]
)
display(composition_model_grid)


In [ ]:
composition_single_primary_results: dict[str, dict[str, object]] = {}
composition_single_expanded_results: dict[str, dict[str, object]] = {}

if RUN_COMPOSITION_GRID:
    for col, formula in composition_single_primary_formulas.items():
        term = composition_terms[col]
        print(f"\nFitting primary single-predictor composition model: {col}")
        composition_single_primary_results[col] = fit_and_summarise_model(
            data=composition_primary_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                term,
                "1|window_idx_sigma",
                "1|clade_sigma",
                "1|cluster_id_sigma",
            ],
            odds_ratio_vars=[term],
            categorical=COMP_GROUP_VARS,
            **SAMPLING_KWARGS,
        )

    for col, formula in composition_single_expanded_formulas.items():
        term = composition_terms[col]
        print(f"\nFitting expanded single-predictor composition model: {col}")
        composition_single_expanded_results[col] = fit_and_summarise_model(
            data=composition_expanded_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                term,
                *EXPANDED_ADJUSTERS,
                "1|window_idx_sigma",
                "1|clade_sigma",
                "1|cluster_id_sigma",
            ],
            odds_ratio_vars=[term],
            categorical=COMP_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_COMPOSITION_GRID=False; skipping single-predictor composition sampling.")

composition_single_results = composition_single_primary_results

In [ ]:
composition_joint_primary_results: dict[str, dict[str, object]] = {}
composition_joint_expanded_results: dict[str, dict[str, object]] = {}

if RUN_COMPOSITION_GRID:
    print("\nFitting joint primary composition model")
    composition_joint_primary_results["joint_primary"] = fit_and_summarise_model(
        data=composition_primary_fit_df,
        formula=joint_composition_primary_formula,
        var_names=[
            "Intercept",
            *composition_terms.values(),
            "1|window_idx_sigma",
            "1|clade_sigma",
            "1|cluster_id_sigma",
        ],
        odds_ratio_vars=list(composition_terms.values()),
        categorical=COMP_GROUP_VARS,
        **SAMPLING_KWARGS,
    )

    print("\nFitting joint expanded composition model")
    composition_joint_expanded_results["joint_expanded"] = fit_and_summarise_model(
        data=composition_expanded_fit_df,
        formula=joint_composition_expanded_formula,
        var_names=[
            "Intercept",
            *composition_terms.values(),
            *EXPANDED_ADJUSTERS,
            "1|window_idx_sigma",
            "1|clade_sigma",
            "1|cluster_id_sigma",
        ],
        odds_ratio_vars=list(composition_terms.values()),
        categorical=COMP_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    print("RUN_COMPOSITION_GRID=False; skipping joint composition-model sampling.")

joint_composition_results = composition_joint_primary_results


## Save Bayesian Model Results

The cell below writes model grids, fitting-frame summaries, posterior summaries, odds-ratio summaries, sampler diagnostics, and optional ArviZ `InferenceData` files to `sse_detection/results/bayesian_socio_geo_demo`.


In [ ]:
def safe_result_name(value: object) -> str:
    """Return a filesystem-safe name for a model or predictor label."""
    cleaned = re.sub(r"[^A-Za-z0-9_.=-]+", "_", str(value)).strip("_")
    return cleaned or "model"


def posterior_odds_ratio_summary(
    idata: az.InferenceData,
    var_names: Sequence[str],
    *,
    hdi_prob: float = 0.95,
) -> pd.DataFrame:
    """Summarise exponentiated fixed effects and posterior direction probabilities."""
    rows: list[dict[str, object]] = []
    available = set(idata.posterior.data_vars)  # type: ignore

    for var in var_names:
        if var not in available:
            continue

        beta = idata.posterior[var]  # type: ignore
        coefficient_dims = [dim for dim in beta.dims if dim not in {"chain", "draw"}]
        if coefficient_dims:
            beta_iter = beta.stack(__coefficient__=coefficient_dims)
            selectors = range(beta_iter.sizes["__coefficient__"])
        else:
            beta_iter = beta
            selectors = [None]

        for selector in selectors:
            if selector is None:
                beta_component = beta_iter
                parameter = var
            else:
                beta_component = beta_iter.isel(__coefficient__=selector)
                coord = beta_component["__coefficient__"].item()
                if isinstance(coord, tuple):
                    suffix = ", ".join(str(item) for item in coord)
                else:
                    suffix = str(coord)
                parameter = f"{var}[{suffix}]"

            beta_samples = np.asarray(beta_component).reshape(-1)
            odds_ratio_samples = np.exp(beta_samples)
            hdi = az.hdi(odds_ratio_samples, hdi_prob=hdi_prob)
            rows.append(
                {
                    "term": var,
                    "parameter": parameter,
                    "mean": float(np.mean(odds_ratio_samples)),
                    "sd": float(np.std(odds_ratio_samples, ddof=1)),
                    f"hdi_{(1 - hdi_prob) / 2:.1%}": float(hdi[0]),
                    f"hdi_{1 - (1 - hdi_prob) / 2:.1%}": float(hdi[1]),
                    "p_beta_gt_0": float(np.mean(beta_samples > 0)),
                    "p_beta_lt_0": float(np.mean(beta_samples < 0)),
                    "p_or_gt_1": float(np.mean(odds_ratio_samples > 1)),
                    "p_or_lt_1": float(np.mean(odds_ratio_samples < 1)),
                    "n_samples": int(odds_ratio_samples.size),
                }
            )

    return pd.DataFrame(rows)


def sampler_diagnostics_summary(
    idata: az.InferenceData,
    *,
    rhat_threshold: float = 1.01,
    ess_threshold: int = 400,
) -> pd.DataFrame:
    """Create the same core sampler diagnostics used in printed model summaries."""
    rows: list[dict[str, object]] = []
    diagnostic_summary = az.summary(idata)

    if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:  # type: ignore
        n_div = int(idata.sample_stats["diverging"].sum().item())  # type: ignore
        n_total = int(idata.sample_stats["diverging"].size)  # type: ignore
        rows.append(
            {
                "diagnostic": "divergences",
                "value": n_div,
                "total": n_total,
                "status": "OK" if n_div == 0 else "WARNING",
            }
        )

    try:
        bfmi = np.asarray(az.bfmi(idata))
        rows.append(
            {
                "diagnostic": "bfmi_min",
                "value": float(np.nanmin(bfmi)),
                "total": np.nan,
                "status": "OK" if float(np.nanmin(bfmi)) >= 0.3 else "WARNING",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "diagnostic": "bfmi_min",
                "value": np.nan,
                "total": np.nan,
                "status": f"NA: {exc}",
            }
        )

    if "r_hat" in diagnostic_summary.columns:
        max_rhat = float(diagnostic_summary["r_hat"].max(skipna=True))
        rows.append(
            {
                "diagnostic": "max_rhat",
                "value": max_rhat,
                "total": np.nan,
                "status": "OK" if max_rhat <= rhat_threshold else "WARNING",
            }
        )

    if "ess_bulk" in diagnostic_summary.columns:
        min_bulk_ess = float(diagnostic_summary["ess_bulk"].min(skipna=True))
        rows.append(
            {
                "diagnostic": "min_bulk_ess",
                "value": min_bulk_ess,
                "total": np.nan,
                "status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
            }
        )

    if "ess_tail" in diagnostic_summary.columns:
        min_tail_ess = float(diagnostic_summary["ess_tail"].min(skipna=True))
        rows.append(
            {
                "diagnostic": "min_tail_ess",
                "value": min_tail_ess,
                "total": np.nan,
                "status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
            }
        )

    return pd.DataFrame(rows)


def write_summary_table(summary: pd.DataFrame, path: Path) -> None:
    """Write an ArviZ summary while preserving parameter names from the index."""
    out = summary.copy()
    out.insert(0, "parameter", out.index.astype(str))
    out.to_csv(path, index=False)


def save_model_result(
    result: dict[str, object],
    *,
    domain: str,
    model_set: str,
    predictor: str,
    output_dir: Path = RESULT_DIR,
    save_idata: bool = SAVE_INFERENCE_DATA,
) -> dict[str, object]:
    """Save one fitted model result and return a manifest row."""
    model_dir = output_dir / domain / model_set / safe_result_name(predictor)
    model_dir.mkdir(parents=True, exist_ok=True)

    summary = result.get("summary")
    if isinstance(summary, pd.DataFrame):
        write_summary_table(summary, model_dir / "posterior_summary.csv")

    idata = result.get("idata")
    odds_ratio_vars = result.get("odds_ratio_vars", [])
    if isinstance(idata, az.InferenceData):
        odds_ratio_summary = posterior_odds_ratio_summary(idata, odds_ratio_vars)  # type: ignore[arg-type]
        if not odds_ratio_summary.empty:
            odds_ratio_summary.to_csv(model_dir / "odds_ratio_summary.csv", index=False)

        diagnostics = sampler_diagnostics_summary(idata)
        diagnostics.to_csv(model_dir / "sampler_diagnostics.csv", index=False)

        if save_idata:
            idata.to_netcdf(model_dir / "idata.nc")  # type: ignore

    metadata = pd.DataFrame(
        [
            {
                "domain": domain,
                "model_set": model_set,
                "predictor": predictor,
                "formula": result.get("formula"),
                "n_rows": result.get("n_rows"),
                "candidate_rate": result.get("candidate_rate"),
                "use_dev_sample": USE_DEV_SAMPLE,
            }
        ]
    )
    metadata.to_csv(model_dir / "metadata.csv", index=False)

    return {
        "domain": domain,
        "model_set": model_set,
        "predictor": predictor,
        "model_dir": str(model_dir.relative_to(PROJECT_ROOT)),
        "n_rows": result.get("n_rows"),
        "candidate_rate": result.get("candidate_rate"),
    }


RESULT_DIR.mkdir(parents=True, exist_ok=True)

composition_model_grid.to_csv(RESULT_DIR / "composition_model_grid.csv", index=False)
mixing_model_grid.to_csv(RESULT_DIR / "mixing_model_grid.csv", index=False)

model_frame_summary = pd.concat(
    [composition_fit_frame_summary, mixing_fit_frame_summary],
    ignore_index=True,
)
model_frame_summary.to_csv(RESULT_DIR / "model_frame_summary.csv", index=False)

run_config = pd.DataFrame(
    [
        {
            "use_dev_sample": USE_DEV_SAMPLE,
            "composition_dev_sample_rows": COMPOSITION_DEV_SAMPLE_ROWS,
            "mixing_dev_sample_rows": MIXING_DEV_SAMPLE_ROWS,
            "mixing_positive_fraction": MIXING_POSITIVE_FRACTION,
            "composition_positive_fraction": COMPOSITION_POSITIVE_FRACTION,
            "run_composition_grid": RUN_COMPOSITION_GRID,
            "run_mixing_grid": RUN_MIXING_GRID,
            "composition_grid_predictors": ",".join(COMPOSITION_GRID_PREDICTORS),
            "mixing_grid_predictors": ",".join(MIXING_GRID_PREDICTORS),
            "sampling_kwargs": repr(SAMPLING_KWARGS),
            "save_inference_data": SAVE_INFERENCE_DATA,
        }
    ]
)
run_config.to_csv(RESULT_DIR / "run_config.csv", index=False)

result_groups = [
    ("mixing", "single_primary", single_primary_results),
    ("mixing", "single_expanded", single_expanded_results),
    ("mixing", "joint_primary", joint_primary_results),
    ("mixing", "joint_expanded", joint_expanded_results),
    ("composition", "single_primary", composition_single_primary_results),
    ("composition", "single_expanded", composition_single_expanded_results),
    ("composition", "joint_primary", composition_joint_primary_results),
    ("composition", "joint_expanded", composition_joint_expanded_results),
]

manifest_rows = []
for domain, model_set, results in result_groups:
    for predictor, result in results.items():
        manifest_rows.append(
            save_model_result(
                result,
                domain=domain,
                model_set=model_set,
                predictor=predictor,
            )
        )

saved_model_manifest = pd.DataFrame(
    manifest_rows,
    columns=[
        "domain",
        "model_set",
        "predictor",
        "model_dir",
        "n_rows",
        "candidate_rate",
    ],
)
saved_model_manifest.to_csv(RESULT_DIR / "saved_model_manifest.csv", index=False)

print(
    f"Saved Bayesian socio-geo-demo outputs to: {RESULT_DIR.relative_to(PROJECT_ROOT)}"
)
display(saved_model_manifest)